In [9]:
import functools
import os
import sys
import json

from ultralytics import YOLO
import pandas as pd

sys.path.append("..")
import ultralytics_validator_wrapper
from tflite_inference import TFLiteInferer, TFLiteMerged
from tflite_inference_baseline import TFLiteMerged as TFLiteInfererCorrected

pd.options.display.float_format = '{:.3f}'.format

%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
BASE_EXPERIMENT_DIR = "../results/cam3_revB_batch1_5_margin_15_rq_bg_wb_alif_clahe_box_01_dfl_01_2nd"
BASE_WEIGHTS_DIR = os.path.join(BASE_EXPERIMENT_DIR, "weights")
WEIGHTS_PTH = os.path.join(BASE_WEIGHTS_DIR, "best.pt")

HOLDOUT_PTH = "/home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/project_holdout.yaml"

if os.path.exists(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json")):
    with open(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json"), "r") as f:
        data = json.load(f)
    CONF_TH = float(data["conf_th"])
    IOU_TH = float(data["iou_th"])
else:
    print("No inference_params.json found, using default thresholds.")
    CONF_TH = 0.35
    IOU_TH = 0.50

print(f"Using CONF_TH={CONF_TH}, IOU_TH={IOU_TH}")

args = dict(
    task="pose",
    device=[0],
    batch=128,
    verbose=False,
    conf=CONF_TH,
    iou=IOU_TH,
    imgsz=320,
    model="val",
    rect=False,
    single_cls=False,
)

args_reduced_iou = dict(
    task="pose",
    device=[0],
    batch=128,
    verbose=False,
    conf=CONF_TH,
    iou=0.35,
    imgsz=320,
    model="val",
    rect=False,
    single_cls=False,
)

Using CONF_TH=0.2, IOU_TH=0.5


In [11]:
model = YOLO(WEIGHTS_PTH)

In [12]:
inferer = TFLiteInferer(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_saved_model/best_full_integer_quant.tflite"),
    head_pth=os.path.join(BASE_WEIGHTS_DIR, "head_saved_model/head_float32.tflite"),
    conf_th=CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

inferer_merged = TFLiteMerged(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_non_separated_saved_model_full_npu/best_full_integer_quant.tflite"),
    conf_th=CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

inferer_merged_iou = TFLiteMerged(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_non_separated_saved_model_full_npu/best_full_integer_quant.tflite"),
    conf_th=CONF_TH,
    iou_th=0.35,
    imgsz=320,
)

inferer_merged_bias = TFLiteInfererCorrected(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_non_separated_saved_model_full_npu/best_full_integer_quant.tflite"),
    conf_th=CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
    bias_pth="ridge_regression_params_tiny.json",
)

inferer_merged_bias_iou = TFLiteInfererCorrected(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_non_separated_saved_model_full_npu/best_full_integer_quant.tflite"),
    conf_th=CONF_TH,
    iou_th=0.35,
    imgsz=320,
    bias_pth="ridge_regression_params_tiny.json",
)

In [13]:
args["data"] = HOLDOUT_PTH
args_reduced_iou["data"] = HOLDOUT_PTH

print("BASELINE")
ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

print("MERGED")
ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_merged, args),
    False,
)

print("MERGED REDUCED IOU")
ultralytics_validator_wrapper.validate_model(
    args_reduced_iou,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_merged_iou, args_reduced_iou),
    False,
)

BASELINE


val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]


tst validation: 100%|██████████| 2/2 [00:05<00:00,  2.94s/it]


                   all        148        409       0.94      0.834      0.914      0.662      0.967      0.858      0.938      0.933
   task    F1     P     R  MAP@50
0   box 0.884 0.940 0.834   0.914
1  pose 0.909 0.967 0.858   0.938
MERGED


val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]
tst validation: 100%|██████████| 2/2 [00:05<00:00,  2.93s/it]


                   all        148        409      0.663      0.753      0.683      0.219      0.795      0.851      0.851      0.843
   task    F1     P     R  MAP@50
0   box 0.705 0.663 0.753   0.683
1  pose 0.822 0.795 0.851   0.851
MERGED REDUCED IOU


val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]
tst validation: 100%|██████████| 2/2 [00:05<00:00,  2.90s/it]


                   all        148        409      0.756      0.707       0.71      0.218      0.947      0.851      0.921      0.912
   task    F1     P     R  MAP@50
0   box 0.730 0.756 0.707   0.710
1  pose 0.896 0.947 0.851   0.921


In [14]:
print("MERGED BIAS")
ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_merged_bias, args),
    False,
)

print("MERGED BIAS REDUCED IOU")
ultralytics_validator_wrapper.validate_model(
    args_reduced_iou,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_merged_bias_iou, args_reduced_iou),
    False,
)

MERGED BIAS


val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]


tst validation: 100%|██████████| 2/2 [00:05<00:00,  2.90s/it]


                   all        148        409       0.31      0.306      0.223     0.0481      0.961      0.852      0.925      0.916
   task    F1     P     R  MAP@50
0   box 0.308 0.310 0.306   0.223
1  pose 0.903 0.961 0.852   0.925
MERGED BIAS REDUCED IOU


val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]
tst validation: 100%|██████████| 2/2 [00:05<00:00,  2.85s/it]


                   all        148        409      0.308      0.289      0.227     0.0495      0.962      0.869      0.937      0.928
   task    F1     P     R  MAP@50
0   box 0.298 0.308 0.289   0.227
1  pose 0.913 0.962 0.869   0.937
